In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm.auto import tqdm
from utils import rnmse
from torch.utils.data import DataLoader, TensorDataset
from sklearn.base import BaseEstimator
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import StandardScaler
from torch.utils.data import Subset
from skorch import NeuralNet
from neuralop import FNO
import pandas as pd
import seaborn as sns
import matplotlib
sns.set_style('whitegrid')
sns.set_context('notebook')
sns.set_palette('hot', n_colors=7)
plt.rc('text', usetex=True)      

font = {'family' : 'serif',
        'weight' : 'bold',
        'size'   : 22}

matplotlib.rc('font', **font)

plt.rc('xtick',labelsize=12)
plt.rc('ytick',labelsize=12)

X = torch.cat([torch.load("../dataset/x_train.pt"), torch.load("../dataset/x_train_2.pt")])
y = torch.cat([torch.load("../dataset/y_train.pt"), torch.load("../dataset/y_train_2.pt")])
c = torch.cat([torch.load("../dataset/c_train.pt"), torch.load("../dataset/c_train_2.pt")])

X_test = torch.load("../dataset/x_test.pt")
y_test = torch.load("../dataset/y_test.pt")
c_test = torch.load("../dataset/c_test.pt")

class ResidualRegressor(BaseEstimator):
    def __init__(self, regressor, n_components=64):
        self.n_components = n_components
        self.pipe_lm = TransformedTargetRegressor(
            regressor=Pipeline([
                ("pca", PCA(n_components=self.n_components)),
                ("lm", LinearRegression(n_jobs=-1))
            ]),
            transformer=PCA(n_components=self.n_components),
            check_inverse=False
        )
        self.regressor = regressor

    def fit(self, X, y):
        X_np, y_np = X.numpy().reshape((X.shape[0], -1)), y.numpy().reshape((y.shape[0], -1))
        self.pipe_lm.fit(X_np, y_np)
        y_hat_lm = torch.tensor(self.pipe_lm.predict(X_np).reshape(y.shape))
        self.regressor.fit({'x': X, 'x_guess': y_hat_lm}, y)

    def predict(self, X):
        return torch.tensor(self.regressor.predict({'x': X, 'x_guess': torch.tensor(self.pipe_lm.predict(X.reshape((X.shape[0], -1))).reshape(X.shape))}))

def scorer_rnmse(estimator, x, y):
    return rnmse(estimator.predict(x), y)

n_points = X.shape[0]
n_cv = 5
n_epochs = 500

scores = {}

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [14]:
import neuralop
from neuralop.layers.fno_block import FNOBlocks

In [29]:
class NO(nn.Module):
    def __init__(self, input_dim=(X.shape[1:]), output_dim=(y.shape[1:])):
        super().__init__()

        self.features = TFNO(
           n_modes=(32, 32),
           hidden_channels=16,
           in_channels=1,
           out_channels=1,
           factorization='tucker',
           implementation='factorized',
           rank=0.1
        )
        self.alpha, self.beta, self.gamma = nn.Parameter(torch.tensor([1/3])), nn.Parameter(torch.tensor([1/3])), nn.Parameter(torch.tensor([1/3]))

    def forward(self, x, x_guess):
        return self.alpha*x_guess + self.beta*x + self.gamma*self.features(x.unsqueeze(1)).squeeze()

device = 'cuda'

def get_rnmse():
    return rnmse

In [27]:
n_points = X.shape[0]

model_2 = ResidualRegressor(regressor=NeuralNet(
            NO,
            max_epochs=n_epochs,
            criterion=get_rnmse,
            optimizer=optim.Adam,
            lr=1e-3,
            iterator_train__shuffle=False,
            device=device,
    ), n_components=512)
model.fit({'x': X[:n_points], 'x_guess': X[:n_points]}, y[:n_points])

In [ ]:
model_2 = ResidualRegressor(regressor=NeuralNet(
            NO,
            max_epochs=n_epochs,
            criterion=get_rnmse,
            optimizer=optim.Adam,
            lr=1e-3,
            iterator_train__shuffle=False,
            device=device,
    ), n_components=512)

pd.DataFrame({'NO': cross_val_score(
    model_2,
    X[:n_points], y[:n_points],
    cv=n_cv,
    scoring=scorer_rnmse,
    verbose=1
)}).to_csv('scores_no.csv')

  epoch    train_loss    valid_loss      dur
-------  ------------  ------------  -------
      1        0.2393        0.1586  75.0566
      2        0.1279        0.1075  74.8224
      3        0.0999        0.0959  75.1510
      4        0.0916        0.0898  76.0138
      5        0.0870        0.0861  77.1986
      6        0.0839        0.0835  76.3754
      7        0.0817        0.0817  75.1865
      8        0.0800        0.0803  75.8241
      9        0.0787        0.0791  76.1273
     10        0.0775        0.0781  79.7792
     11        0.0767        0.0774  77.9619
     12        0.0760        0.0768  78.1675
     13        0.0754        0.0763  76.1003
     14        0.0749        0.0759  77.8808
     15        0.0745        0.0756  77.6014
     16        0.0741        0.0753  81.1211
     17        0.0738        0.0751  79.4865
     18        0.0735        0.0749  81.3646
     19        0.0733        0.0747  80.1763
     20        0.0730        0.0745  79.6217
     21   